# Snapget-12 — train model xác minh ảnh cho AI Daily Quest

Notebook **mỏng**: mọi logic nằm trong `ml/snapget12/` + `ml/scripts/` của repo (để review/diff được). Chạy trên **Google Colab (GPU T4)** hoặc Kaggle. Xem `Snapget/.claude/QUEST_AI_PLAN.md` mục 3–5 và `ml/README.md`.

**Trình tự:** 0 cài đặt → 1 tải COCO subset → 2 cache embedding → 3 train head v0 → 4 chọn ngưỡng + đánh giá → 5 export ONNX int8 → **upload lên Space** → (M6) 6 fine-tune v1 → 8 đánh giá v1 + Snapget-12 → 7 ablation OWL-ViT.

> ⚠️ Runtime → Change runtime type → **T4 GPU** trước khi chạy. Bước 1 tải ~7–9GB về disk Colab (không về máy). Mount Drive để giữ `data/embeddings.npz` + `artifacts/` qua các phiên.

In [ ]:
# 0a. Mount Drive (giữ cache embedding + artifact qua các phiên Colab)
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/snapget-ml'   # đổi nếu muốn
import os; os.makedirs(WORK, exist_ok=True)

In [ ]:
# 0b. Lấy code từ repo (đổi URL/branch cho đúng) + cài thư viện
%cd /content
!rm -rf datn && git clone --depth 1 https://github.com/<user>/<repo>.git datn
%cd /content/datn/ml
!pip install -q -r requirements.txt
!python -c "import torch, torchvision, fiftyone; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
# 0c. Trỏ data/artifacts vào Drive (symlink) để không mất khi Colab reset
!mkdir -p {WORK}/data {WORK}/artifacts
!ln -sfn {WORK}/data data && ln -sfn {WORK}/artifacts artifacts
!ls -la

## 1. Tải COCO 2017 subset (12 lớp + negative) → `data/manifest.csv`
Chỉ chạy lần đầu (~20–40 phút tuỳ mạng). Giảm `--max-per-class`/`--negatives` nếu hết disk.

In [ ]:
!python scripts/01_prepare_coco.py --out data/manifest.csv --max-per-class 4000 --negatives 12000 --test-max 5000
!head -3 data/manifest.csv && wc -l data/manifest.csv

## 2. Cache embedding backbone đóng băng (chạy 1 lần, ~10 phút GPU)

In [ ]:
!python scripts/02_cache_embeddings.py --manifest data/manifest.csv --out data/embeddings.npz --batch 128

## 3. Train head **v0** từ cache (~1–2 phút)

In [ ]:
!python scripts/03_train_head.py --emb data/embeddings.npz --out artifacts/v0/head.pt --epochs 30

## 4. Chọn ngưỡng per-class (precision ≥ 0.8, max recall) + đánh giá COCO val/test + PR curve

In [ ]:
!python scripts/04_eval_thresholds.py --emb data/embeddings.npz --head artifacts/v0/head.pt --out artifacts/v0
from IPython.display import Image, display
display(Image('artifacts/v0/pr_curves_test.png'))

## 5. Export ONNX int8 + `model_meta.json` → **upload lên HF Model repo** (miễn phí)
Cần token HF quyền **write**. Điền `MODEL_REPO` = `<user>/snapget-ai-model`. Sau đó AI service trên **Cloud Run** tự tải model qua env `MODEL_REPO` (QUEST_AI_PLAN mục 17.C5b).

In [ ]:
from huggingface_hub import login
login()   # dán token write
MODEL_REPO = '<user>/snapget-ai-model'   # HF MODEL repo (mien phi) — Cloud Run tai tu day
!python scripts/05_export_onnx.py --head artifacts/v0/head.pt --version v0 --out artifacts/v0 --upload {MODEL_REPO}

✅ **Tới đây là xong M1+M2 phần model**: deploy/restart Cloud Run với `MODEL_REPO` (plan 17.C5b) rồi `GET <Service URL>/health` phải trả `modelVersion: "v0"`, `verifierReady: true`.

---
## M6 — Snapget-12, fine-tune v1, ablation

### 6a. Upload tập **Snapget-12** tự chụp
Cấu trúc thư mục (zip rồi kéo lên Drive `{WORK}/data/snapget12/`): `snapget12/cup/*.jpg`, `snapget12/bottle/*.jpg`, …, `snapget12/negative/*.jpg`. Ảnh có 2 vật thể → đặt tên `cup+book_01.jpg`. Mỗi lớp ~10 ảnh/thành viên, chụp **bằng chính app/điện thoại** (dọc, trong nhà, đèn vàng…).

In [ ]:
# 6b. Đánh giá model v0 ĐANG CHẠY (ONNX int8) trên Snapget-12 -> domain shift so với COCO test
#     (báo cáo: 3 lớp book/backpack/keyboard dự kiến thấp nhất -> lý do app chỉ ra đề 9 lớp, plan 2.3)
!python scripts/08_eval_images.py --onnx artifacts/v0/model.onnx --thresholds artifacts/v0/thresholds.json \
    --manifest data/manifest.csv --snapget12 data/snapget12 --out artifacts/v0

### 6c. Fine-tune **v1** (mở block conv cuối, augmentation crop dọc 3:4) — ~30–60 phút T4

In [ ]:
!python scripts/06_finetune_v1.py --manifest data/manifest.csv --init-head artifacts/v0/head.pt --out artifacts/v1/model.pt --epochs 8
# chọn ngưỡng trên val + đánh giá test + Snapget-12 (cùng tập với v0 để so sánh delta)
!python scripts/08_eval_images.py --full artifacts/v1/model.pt --manifest data/manifest.csv \
    --choose-thresholds-on val --snapget12 data/snapget12 --out artifacts/v1

### 6d. Export v1 → upload Model repo (thay v0) — chỉ khi v1 tốt hơn trên **Snapget-12** (rồi restart Cloud Run)

In [ ]:
!python scripts/05_export_onnx.py --full artifacts/v1/model.pt --version v1 --out artifacts/v1 --upload {MODEL_REPO}

### 7. Ablation: OWL-ViT zero-shot trên cùng 2 tập test (~150M params, vài giây/ảnh CPU)

In [ ]:
!python scripts/07_ablation_owlvit.py --manifest data/manifest.csv --snapget12 data/snapget12 --out artifacts/owlvit --max-test 1500

### 8. Bảng tổng hợp cho báo cáo

In [ ]:
import json, glob
rows = []
for name in ['v0', 'v1', 'owlvit']:
    for split in ['test', 'snapget12']:
        p = f'artifacts/{name}/metrics_{split}.json'
        try:
            m = json.load(open(p))
            rows.append((name, split, m['n'], m['mAP'], m['macroF1'], m.get('latencyMs')))
        except FileNotFoundError:
            pass
print(f"{'model':8s} {'split':10s} {'n':>6s} {'mAP':>7s} {'macroF1':>8s}  latency")
for r in rows:
    print(f"{r[0]:8s} {r[1]:10s} {r[2]:6d} {r[3]:7.4f} {r[4]:8.4f}  {r[5]}")